# Chat With Your Own Data — RAG Q&A System
**Shamiema Shahzad**

Pipeline: PDF → chunks → embeddings → FAISS → retrieve top-k → LLM answers from context only.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Install libraries

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface \
    sentence-transformers faiss-cpu pypdf transformers accelerate

## 2. Upload a PDF

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick your PDF
pdf_path = list(uploaded.keys())[0]
print("Loaded:", pdf_path)

## 3. Extract text and split into chunks
We use ~1000 characters (~150-200 words) with overlap. The embedding model reads max ~256 tokens, so much bigger chunks would be cut off.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader(pdf_path).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(pages)
print(f"{len(pages)} pages -> {len(chunks)} chunks")
print(chunks[0].page_content[:300])

## 4. Embeddings + FAISS vector store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print("Vector store ready")

## 5. Load the LLM (open-source, free, ungated)
Qwen2.5-3B-Instruct fits on the free T4 GPU. You can swap in a Mistral/Llama model later.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
generator = pipeline("text-generation", model=model, tokenizer=tok)
print("LLM loaded")

## 6. Question-answering function

In [ ]:
def ask(question, show_sources=True):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)

    messages = [
        {"role": "system", "content":
         "You answer questions using ONLY the provided context. "
         "If the answer is not in the context, say: 'I could not find this in the document.'"},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = generator(prompt, max_new_tokens=300, do_sample=False, return_full_text=False)
    answer = out[0]["generated_text"].strip()

    print("Q:", question)
    print("A:", answer)
    if show_sources:
        print("Sources (pages):", sorted({d.metadata.get("page", 0) + 1 for d in docs}))
    print("-" * 60)

## 7. Demo — ask 3-4 questions

In [ ]:
ask("What is this document about?")
ask("Summarize the main points.")
ask("Replace this with a specific question from your PDF")
ask("What is the capital of Mars?")   # not in doc -> should say it can't find it

## 8. Interactive chat (optional)

In [ ]:
while True:
    q = input("Ask (or type 'exit'): ")
    if q.lower() == "exit":
        break
    ask(q)